In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Use TensorFlow 2.x Keras API
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import regularizers

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version: {tf.__version__}")

/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow version: 2.20.0


In [3]:
# Load the cleaned fighter data and match data
# Note: You'll need to download the match data from Kaggle or create it
try:
    df_fighters = pd.read_csv("fighters_clean.csv")
    print(f"Loaded {len(df_fighters)} fighters")
except FileNotFoundError:
    print("Error: fighters_clean.csv not found. Please run UFC_Cleaning.ipynb first.")
    df_fighters = pd.DataFrame()

try:
    df_matches = pd.read_csv("matches_clean.csv")
    print(f"Loaded {len(df_matches)} matches")
except FileNotFoundError:
    print("Warning: matches_clean.csv not found.")
    print("You'll need to download the UFC match data from Kaggle:")
    print("https://www.kaggle.com/rajeevw/ufcdata")
    print("Or create matches_clean.csv with columns: Red, Blue, Winner, Weight Class")
    df_matches = pd.DataFrame()

Loaded 4444 fighters
Loaded 8263 matches


In [4]:
df_fighters.set_index("Name", inplace=True)
df_fighters.head()

,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Wins,Losses,Draws,Win Percentage
Name,,,,,,,,,,,,,,,,,
Tom Aaron,No Name,155,47.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,5,3,0,0.62
Danny Abbadi,The Assassin,155,42.5,3.29,0.38,4.41,0.57,0.00,0.00,0.77,0.0,"5' 11""",0,4,6,0,0.40
Nariman Abbasov,Bayraktar,155,31.9,3.00,0.20,5.67,0.46,0.00,0.00,0.66,0.0,"5' 8""",66,28,4,0,0.88
Darion Abbey,No Name,265,32.9,8.44,0.50,14.06,0.28,0.00,0.00,0.00,0.0,"6' 2""",80,9,5,0,0.64
David Abbott,Tank,265,60.7,1.35,0.30,3.55,0.38,1.07,0.33,0.66,0.0,"6' 0""",0,10,15,0,0.40


In [5]:
df_fighters['Total Fights'] = df_fighters['Wins'] + df_fighters['Losses'] + df_fighters['Draws']
df_fighters

,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Wins,Losses,Draws,Win Percentage,Total Fights
Name,,,,,,,,,,,,,,,,,,
Tom Aaron,No Name,155,47.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,5,3,0,0.62,8
Danny Abbadi,The Assassin,155,42.5,3.29,0.38,4.41,0.57,0.00,0.00,0.77,0.0,"5' 11""",0,4,6,0,0.40,10
Nariman Abbasov,Bayraktar,155,31.9,3.00,0.20,5.67,0.46,0.00,0.00,0.66,0.0,"5' 8""",66,28,4,0,0.88,32
Darion Abbey,No Name,265,32.9,8.44,0.50,14.06,0.28,0.00,0.00,0.00,0.0,"6' 2""",80,9,5,0,0.64,14
David Abbott,Tank,265,60.7,1.35,0.30,3.55,0.38,1.07,0.33,0.66,0.0,"6' 0""",0,10,15,0,0.40,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Alex Zuniga,No Name,145,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,6,3,0,0.67,9
George Zuniga,No Name,185,NaN,7.64,0.38,5.45,0.37,0.00,0.00,1.00,0.0,"5' 9""",0,3,1,0,0.75,4
Allan Zuniga,Tigre,155,33.8,3.93,0.52,1.80,0.61,0.00,0.00,0.57,1.0,"5' 7""",70,13,1,0,0.93,14


In [6]:
# Drop columns that we don't need for training (only if they exist)
cols_to_drop = ["Weight Class", "Wins", "Losses", "Draws"]
cols_to_drop = [col for col in cols_to_drop if col in df_fighters.columns]
if cols_to_drop:
    df_fighters.drop(cols_to_drop, axis=1, inplace=True)
    print(f"Dropped columns: {cols_to_drop}")
else:
    print("No columns to drop (they may have already been removed)")


Dropped columns: ['Wins', 'Losses', 'Draws']


In [7]:
df_fighters.head(1)

,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Win Percentage,Total Fights
Name,,,,,,,,,,,,,,,
Tom Aaron,No Name,155,47.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0.62,8


In [8]:
# Get only numeric columns for stats (exclude Name, Nickname, and other non-numeric columns)
stats = [col for col in df_fighters.columns if df_fighters[col].dtype in ['int64', 'float64']]
# Also include Height: if it exists (will convert to numeric later if needed)
if 'Height:' in df_fighters.columns and 'Height:' not in stats:
    stats.append('Height:')
stats

['Weight:',
 'Age',
 'SLpM:',
 'Str. Acc.:',
 'SApM:',
 'Str. Def:',
 'TD Avg.:',
 'TD Acc.:',
 'TD Def.:',
 'Sub. Avg.:',
 'Reach:',
 'Win Percentage',
 'Total Fights',
 'Height:']

In [9]:
df_matches.head(4)

,Red,Blue,Winner,Weight Class
0,Brandon Royval,Manel Kape,Manel Kape,Flyweight
1,Giga Chikadze,Kevin Vallejos,Kevin Vallejos,Featherweight
2,Cesar Almeida,Cezary Oleksiejczuk,Cezary Oleksiejczuk,Middleweight
3,Melquizael Costa,Morgan Charriere,Melquizael Costa,Featherweight


In [10]:
# Drop Weight Class column if it exists
if "Weight Class" in df_matches.columns:
    df_matches.drop("Weight Class", axis=1, inplace=True)
    print("Dropped 'Weight Class' column from matches")
else:
    print("'Weight Class' column not found in matches (may have already been removed)")

Dropped 'Weight Class' column from matches


In [11]:
df_matches["result"] = (df_matches["Winner"] == df_matches["Red"]).astype("int")
df_matches.drop("Winner", axis=1, inplace=True)

In [12]:
df_matches.head()

# if df_matches.loc[df_matches['result']==1:
#     for i =10 to 570
#         df_red = 
    
# idx = (df_matches['result'] == 1)

,Red,Blue,result
0,Brandon Royval,Manel Kape,0
1,Giga Chikadze,Kevin Vallejos,0
2,Cesar Almeida,Cezary Oleksiejczuk,0
3,Melquizael Costa,Morgan Charriere,1
4,Kennedy Nzechukwu,Marcus Buchecha,0


In [13]:
# make sure all fighters are in the figher database and remove any that aren't
fighters_list = df_fighters.index.values.tolist()

df_matches = df_matches.loc[(df_matches["Red"].isin(fighters_list)) &
                                  (df_matches["Blue"].isin(fighters_list))].copy()

df_fighters.head()

,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Win Percentage,Total Fights
Name,,,,,,,,,,,,,,,
Tom Aaron,No Name,155,47.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,0.62,8
Danny Abbadi,The Assassin,155,42.5,3.29,0.38,4.41,0.57,0.00,0.00,0.77,0.0,"5' 11""",0,0.40,10
Nariman Abbasov,Bayraktar,155,31.9,3.00,0.20,5.67,0.46,0.00,0.00,0.66,0.0,"5' 8""",66,0.88,32
Darion Abbey,No Name,265,32.9,8.44,0.50,14.06,0.28,0.00,0.00,0.00,0.0,"6' 2""",80,0.64,14
David Abbott,Tank,265,60.7,1.35,0.30,3.55,0.38,1.07,0.33,0.66,0.0,"6' 0""",0,0.40,25


In [14]:
# turn df_matches into fighter 1 - fighter 2 (the differences) for training the model
# First, ensure we only use numeric columns and convert Height: to numeric if needed
numeric_stats = []

# Helper function to convert height string to inches
def height_to_inches(height_str):
    """Convert height string like '5\' 11"' to total inches (71.0)"""
    if pd.isna(height_str) or height_str == '':
        return np.nan
    try:
        # Format: "5' 11"" or "5'11""
        parts = str(height_str).replace('"', '').split("'")
        if len(parts) == 2:
            feet = float(parts[0].strip())
            inches = float(parts[1].strip())
            return feet * 12 + inches
        else:
            return np.nan
    except:
        return np.nan

# Process each stat column
for col in stats:
    if col in df_fighters.columns:
        # Check if column is numeric
        if df_fighters[col].dtype in ['int64', 'float64']:
            numeric_stats.append(col)
        elif col == 'Height:':
            # Convert Height: from "5' 11"" format to total inches
            print(f"Converting {col} from feet/inches to total inches...")
            df_fighters[col + '_inches'] = df_fighters[col].apply(height_to_inches)
            numeric_stats.append(col + '_inches')
            # Drop the original Height: column from stats
            if col in numeric_stats:
                numeric_stats.remove(col)
        else:
            print(f"Skipping non-numeric column: {col}")

print(f"Using {len(numeric_stats)} numeric stats columns for training")

# Calculate differences for each numeric stat
for col in numeric_stats:
    def calc_diff(row):
        try:
            red_val = df_fighters.loc[row["Red"], col]
            blue_val = df_fighters.loc[row["Blue"], col]
            # Handle Series (if loc returns multiple values, take first)
            if isinstance(red_val, pd.Series):
                red_val = red_val.iloc[0]
            if isinstance(blue_val, pd.Series):
                blue_val = blue_val.iloc[0]
            # Convert to float and check for NaN
            red_val = float(red_val) if pd.notna(red_val) else 0.0
            blue_val = float(blue_val) if pd.notna(blue_val) else 0.0
            return red_val - blue_val
        except (KeyError, IndexError, ValueError, TypeError):
            return 0.0
    
    df_matches[col] = df_matches.apply(calc_diff, axis=1)
    
# drop fighter1 and fighter2 columns
df_matches.drop(["Red", "Blue"], axis=1, inplace=True)

Converting Height: from feet/inches to total inches...
Using 14 numeric stats columns for training


In [15]:
df_matches.head()

,result,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Reach:,Win Percentage,Total Fights,Height:_inches
0,0,0.0,1.2,0.50,-0.15,0.17,-0.14,0.25,0.45,-0.36,0.8,0.0,-0.11,-3.0,4.0
1,0,0.0,13.3,-2.03,-0.04,-1.24,0.03,-0.53,0.05,-0.18,0.2,6.0,-0.23,3.0,5.0
2,0,0.0,11.8,1.47,-0.28,1.26,-0.01,-5.77,-0.85,-0.48,0.2,-1.0,-0.07,-11.0,-2.0
3,1,0.0,-0.9,0.40,-0.01,-1.45,0.01,-0.13,-0.08,-0.25,0.9,2.0,0.16,-2.0,2.0
4,0,-19.0,-2.4,3.26,-0.03,2.41,0.00,-1.50,0.29,0.80,0.2,6.0,0.05,13.0,3.0


In [16]:
print(len(df_matches.loc[df_matches["result"]==1]))
print(len(df_matches.loc[df_matches["result"]==0]))

5167
3070


In [17]:
df_matches.to_csv(r'training_model.csv', index=False)

In [18]:
X, y = df_matches.iloc[:, 1:], df_matches.iloc[:, 0]

# Check for and handle NaN values
print(f"Shape before cleaning: {X.shape}")
print(f"NaN values per column:\n{X.isna().sum()}")

# Fill NaN values with 0 (or median if preferred)
X = X.fillna(0)

# Ensure all columns are numeric
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

print(f"Shape after cleaning: {X.shape}")
print(f"Data types:\n{X.dtypes}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

# Save the scaler for later use in predictions
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved to scaler.pkl")

Shape before cleaning: (8237, 14)
NaN values per column:
Weight:           0
Age               0
SLpM:             0
Str. Acc.:        0
SApM:             0
Str. Def:         0
TD Avg.:          0
TD Acc.:          0
TD Def.:          0
Sub. Avg.:        0
Reach:            0
Win Percentage    0
Total Fights      0
Height:_inches    0
dtype: int64
Shape after cleaning: (8237, 14)
Data types:
Weight:           float64
Age               float64
SLpM:             float64
Str. Acc.:        float64
SApM:             float64
Str. Def:         float64
TD Avg.:          float64
TD Acc.:          float64
TD Def.:          float64
Sub. Avg.:        float64
Reach:            float64
Win Percentage    float64
Total Fights      float64
Height:_inches    float64
dtype: object
Training set shape: (5765, 14)
Test set shape: (2472, 14)
Scaler saved to scaler.pkl


In [19]:
# Build the neural network model
model = Sequential()

model.add(Dense(16, input_dim=X_train_scaled.shape[1],
                activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(16, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(
    loss='binary_crossentropy', 
    optimizer='adam',  # lowercase 'adam' for TensorFlow 2.x
    metrics=['binary_accuracy']
)

# Display model summary
print("Model Architecture:")
model.summary()

# Train the model with early stopping to prevent overfitting
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Split training data for validation
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42
)

# Train the model
print("\nTraining model...")
history = model.fit(
    x=X_train_fit, 
    y=y_train_fit,
    validation_data=(X_val, y_val),
    epochs=200, 
    batch_size=64, 
    verbose=1,
    callbacks=[early_stopping]
)

# Evaluate on test set
test_results = model.evaluate(x=X_test_scaled, y=y_test, verbose=0)
print(f"\nTest Accuracy = {test_results[1]:.4f} ({test_results[1]*100:.2f}%)")
print(f"Test Loss = {test_results[0]:.4f}")

# Save the model
model.save('ufc_prediction_model.h5')
print("\nModel saved as 'ufc_prediction_model.h5'")

Model Architecture:


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,609 (25.82 KB)

 Trainable params: 6,609 (25.82 KB)

 Non-trainable params: 0 (0.00 B)


Training model...
Epoch 1/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - binary_accuracy: 0.6344 - loss: 2.5844 - val_binary_accuracy: 0.6201 - val_loss: 1.8022
Epoch 2/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step - binary_accuracy: 0.6344 - loss: 1.6241 - val_binary_accuracy: 0.6201 - val_loss: 1.2193
Epoch 3/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 821us/step - binary_accuracy: 0.6385 - loss: 1.1249 - val_binary_accuracy: 0.6461 - val_loss: 0.9248
Epoch 4/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 804us/step - binary_accuracy: 0.6727 - loss: 0.8798 - val_binary_accuracy: 0.6609 - val_loss: 0.7866
Epoch 5/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 754us/step - binary_accuracy: 0.6768 - loss: 0.7641 - val_binary_accuracy: 0.6678 - val_loss: 0.7217
Epoch 6/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 735us/step - binary_accuracy: 0.6804 - loss: 0.7098 - val_binary_accuracy: 0.6748 - val_loss: 0.6911
Epoch 7/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step - binary_accuracy: 0.6805 - loss: 0.6839 - val_binary_accuracy: 0.6704 -


Test Accuracy = 0.6784 (67.84%)
Test Loss = 0.6397

Model saved as 'ufc_prediction_model.h5'


In [20]:
def predict(model, scaler, df_fighters, Red, Blue):
    """Predict the probability of Red fighter winning against Blue fighter."""
    # Get only numeric stats (same as used in training)
    stats = [col for col in df_fighters.columns if df_fighters[col].dtype in ['int64', 'float64']]
    
    # Handle Height: conversion if needed
    if 'Height:' in df_fighters.columns:
        # Convert Height: to inches if not already converted
        if 'Height:_inches' not in df_fighters.columns:
            def height_to_inches(height_str):
                if pd.isna(height_str) or height_str == '':
                    return np.nan
                try:
                    parts = str(height_str).replace('"', '').split("'")
                    if len(parts) == 2:
                        feet = float(parts[0].strip())
                        inches = float(parts[1].strip())
                        return feet * 12 + inches
                    return np.nan
                except:
                    return np.nan
            df_fighters['Height:_inches'] = df_fighters['Height:'].apply(height_to_inches)
        if 'Height:_inches' in df_fighters.columns and 'Height:_inches' not in stats:
            stats.append('Height:_inches')
        if 'Height:' in stats:
            stats.remove('Height:')
    
    data1 = df_fighters.loc[Red, stats]
    data2 = df_fighters.loc[Blue, stats]
    data_diff = (data1 - data2).values.reshape(1, -1)
    data_diff_scaled = scaler.transform(data_diff)
    prediction = model.predict(data_diff_scaled, verbose=0)
    return prediction[0][0]

# Example predictions
try:
    if len(df_fighters) > 0:
        # Try some example predictions (adjust fighter names as needed)
        sample_fighters = df_fighters.index.tolist()[:10]
        if len(sample_fighters) >= 2:
            fighter1 = sample_fighters[0]
            fighter2 = sample_fighters[1]
            prob = predict(model, scaler, df_fighters, fighter1, fighter2)
            print(f"Prediction: {fighter1} vs {fighter2}")
            print(f"Probability of {fighter1} winning: {prob:.2%}")
except Exception as e:
    print(f"Could not run example prediction: {e}")
    print("Make sure you have fighter data loaded and the model is trained.")

Prediction: Tom Aaron vs Danny Abbadi
Probability of Tom Aaron winning: nan%


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [21]:
fight1 = predict(model, scaler, df_fighters, "Israel Adesanya", "Joe Pyfer")
fight2 = predict(model, scaler, df_fighters, "Joe Pyfer", "Israel Adesanya")
print(fight1)
print(fight2)

0.65716267
0.60085124


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [22]:
fighter1_avg = (fight1 + (1 - fight2))/2
fighter1_avg


np.float32(0.5281557)

In [23]:
fighter2_avg = (fight2 + (1 - fight1))/2
fighter2_avg

np.float32(0.4718443)

In [24]:
"""
SAFE MODEL EXPERIMENTATION CELL
Copy this into a new Jupyter notebook cell to experiment without affecting your original model.

This cell:
- Loads your existing model for comparison
- Creates new experimental models with different names
- Saves experimental models with unique filenames
- Compares all models side-by-side
- Never overwrites your original model
"""

# ============================================================================
# SAFE EXPERIMENTATION - Your original model is safe!
# ============================================================================

# First, load your existing model for comparison (optional - comment out if you haven't trained it yet)
try:
    from tensorflow.keras.models import load_model
    original_model = load_model('ufc_prediction_model.h5')
    print("✓ Loaded original model: ufc_prediction_model.h5")
    # Evaluate original model for comparison
    original_test_results = original_model.evaluate(X_test_scaled, y_test, verbose=0)
    print(f"  Original Model - Test Accuracy: {original_test_results[1]:.4f} ({original_test_results[1]*100:.2f}%)")
except:
    print("⚠ Original model not found (that's okay if you haven't trained it yet)")
    original_model = None

print("\n" + "="*80)
print("EXPERIMENTING WITH NEW ARCHITECTURES")
print("="*80)
print()

# Store results for comparison
experiment_results = []

# ============================================================================
# EXPERIMENT 1: Simpler Model (3 layers)
# ============================================================================
print("Experiment 1: Simpler Model (3 layers)")
print("-" * 80)

model_exp1 = Sequential([
    Dense(64, input_dim=X_train_scaled.shape[1], activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dense(16, activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dense(1, activation='sigmoid')
])

model_exp1.compile(loss='binary_crossentropy', optimizer='adam', metrics=['binary_accuracy'])

# Train with validation split
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=0
)

history_exp1 = model_exp1.fit(
    X_train_fit, y_train_fit,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=64,
    verbose=1,  # Set to 0 to hide training progress
    callbacks=[early_stopping]
)

test_results_exp1 = model_exp1.evaluate(X_test_scaled, y_test, verbose=0)
print(f"✓ Test Accuracy: {test_results_exp1[1]:.4f} ({test_results_exp1[1]*100:.2f}%)")
print(f"  Test Loss: {test_results_exp1[0]:.4f}")
print(f"  Epochs Trained: {len(history_exp1.history['loss'])}")

# Save with unique name
model_exp1.save('ufc_model_exp1_simple.h5')
print(f"  Saved as: ufc_model_exp1_simple.h5\n")

experiment_results.append({
    'name': 'Experiment 1: Simple (3 layers)',
    'model': model_exp1,
    'accuracy': test_results_exp1[1],
    'loss': test_results_exp1[0],
    'filename': 'ufc_model_exp1_simple.h5'
})

# ============================================================================
# EXPERIMENT 2: With Dropout
# ============================================================================
print("Experiment 2: With Dropout (3 layers + dropout)")
print("-" * 80)

from tensorflow.keras.layers import Dropout

model_exp2 = Sequential([
    Dense(64, input_dim=X_train_scaled.shape[1], activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dropout(0.3),
    Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dropout(0.3),
    Dense(16, activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dense(1, activation='sigmoid')
])

model_exp2.compile(loss='binary_crossentropy', optimizer='adam', metrics=['binary_accuracy'])

history_exp2 = model_exp2.fit(
    X_train_fit, y_train_fit,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=64,
    verbose=1,
    callbacks=[early_stopping]
)

test_results_exp2 = model_exp2.evaluate(X_test_scaled, y_test, verbose=0)
print(f"✓ Test Accuracy: {test_results_exp2[1]:.4f} ({test_results_exp2[1]*100:.2f}%)")
print(f"  Test Loss: {test_results_exp2[0]:.4f}")
print(f"  Epochs Trained: {len(history_exp2.history['loss'])}")

model_exp2.save('ufc_model_exp2_dropout.h5')
print(f"  Saved as: ufc_model_exp2_dropout.h5\n")

experiment_results.append({
    'name': 'Experiment 2: With Dropout',
    'model': model_exp2,
    'accuracy': test_results_exp2[1],
    'loss': test_results_exp2[0],
    'filename': 'ufc_model_exp2_dropout.h5'
})

# ============================================================================
# EXPERIMENT 3: Wider Layers
# ============================================================================
print("Experiment 3: Wider Layers (128→64→32)")
print("-" * 80)

model_exp3 = Sequential([
    Dense(128, input_dim=X_train_scaled.shape[1], activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dense(64, activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    Dense(1, activation='sigmoid')
])

model_exp3.compile(loss='binary_crossentropy', optimizer='adam', metrics=['binary_accuracy'])

history_exp3 = model_exp3.fit(
    X_train_fit, y_train_fit,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=64,
    verbose=1,
    callbacks=[early_stopping]
)

test_results_exp3 = model_exp3.evaluate(X_test_scaled, y_test, verbose=0)
print(f"✓ Test Accuracy: {test_results_exp3[1]:.4f} ({test_results_exp3[1]*100:.2f}%)")
print(f"  Test Loss: {test_results_exp3[0]:.4f}")
print(f"  Epochs Trained: {len(history_exp3.history['loss'])}")

model_exp3.save('ufc_model_exp3_wide.h5')
print(f"  Saved as: ufc_model_exp3_wide.h5\n")

experiment_results.append({
    'name': 'Experiment 3: Wide Layers',
    'model': model_exp3,
    'accuracy': test_results_exp3[1],
    'loss': test_results_exp3[0],
    'filename': 'ufc_model_exp3_wide.h5'
})

# ============================================================================
# COMPARISON SUMMARY
# ============================================================================
print("="*80)
print("COMPARISON SUMMARY")
print("="*80)
print()

# Add original model if available
if original_model:
    experiment_results.insert(0, {
        'name': 'Original Model (9 layers)',
        'model': original_model,
        'accuracy': original_test_results[1],
        'loss': original_test_results[0],
        'filename': 'ufc_prediction_model.h5'
    })

# Sort by accuracy
experiment_results_sorted = sorted(experiment_results, key=lambda x: x['accuracy'], reverse=True)

print(f"{'Rank':<6} {'Model Name':<35} {'Test Accuracy':<15} {'Test Loss':<12} {'Filename':<30}")
print("-" * 100)
for i, result in enumerate(experiment_results_sorted, 1):
    marker = "🏆" if i == 1 else "  "
    print(f"{marker} {i:<4} {result['name']:<35} {result['accuracy']*100:>6.2f}%        {result['loss']:>6.4f}     {result['filename']:<30}")

print()
print("="*80)
print(f"🏆 BEST MODEL: {experiment_results_sorted[0]['name']}")
print(f"   Accuracy: {experiment_results_sorted[0]['accuracy']*100:.2f}%")
print(f"   Saved as: {experiment_results_sorted[0]['filename']}")
print("="*80)
print()
print("💡 TIP: All models are saved with unique names. Your original model is safe!")
print("   To use a different model, just load it:")
print(f"   model = load_model('{experiment_results_sorted[0]['filename']}')")

# Store the best model in a variable for easy access
best_model = experiment_results_sorted[0]['model']
best_model_name = experiment_results_sorted[0]['filename']

print(f"\n✓ Best model stored in variable: best_model")
print(f"✓ Best model filename: {best_model_name}")


✓ Loaded original model: ufc_prediction_model.h5
  Original Model - Test Accuracy: 0.6784 (67.84%)

EXPERIMENTING WITH NEW ARCHITECTURES

Experiment 1: Simpler Model (3 layers)
--------------------------------------------------------------------------------
Epoch 1/200


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.6054 - loss: 1.4138 - val_binary_accuracy: 0.6479 - val_loss: 1.0799
Epoch 2/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 610us/step - binary_accuracy: 0.6704 - loss: 1.0142 - val_binary_accuracy: 0.6661 - val_loss: 0.8510
Epoch 3/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 588us/step - binary_accuracy: 0.6824 - loss: 0.8162 - val_binary_accuracy: 0.6704 - val_loss: 0.7342
Epoch 4/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 596us/step - binary_accuracy: 0.6887 - loss: 0.7151 - val_binary_accuracy: 0.6704 - val_loss: 0.6752
Epoch 5/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 593us/step - binary_accuracy: 0.6886 - loss: 0.6637 - val_binary_accuracy: 0.6704 - val_loss: 0.6456
Epoch 6/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 622us/step - binary_accuracy: 0.6907 - loss: 0.6373 - val_binary_accuracy: 0.6774 - val_loss: 0.6303
Epoch 7/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 599us/step - binary_accuracy: 0.6890 - loss: 0.6234 - val_binary_accuracy: 0.6800 - val_loss: 0.6222
Epoch 8/200
7

✓ Test Accuracy: 0.6820 (68.20%)
  Test Loss: 0.6005
  Epochs Trained: 200
  Saved as: ufc_model_exp1_simple.h5

Experiment 2: With Dropout (3 layers + dropout)
--------------------------------------------------------------------------------
Epoch 1/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.5558 - loss: 1.4907 - val_binary_accuracy: 0.6331 - val_loss: 1.1841
Epoch 2/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 654us/step - binary_accuracy: 0.6557 - loss: 1.1224 - val_binary_accuracy: 0.6670 - val_loss: 0.9514
Epoch 3/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 691us/step - binary_accuracy: 0.6653 - loss: 0.9201 - val_binary_accuracy: 0.6696 - val_loss: 0.8139
Epoch 4/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 630us/step - binary_accuracy: 0.6608 - loss: 0.8015 - val_binary_accuracy: 0.6748 - val_loss: 0.7322
Epoch 5/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 679us/step - binary_accuracy: 0.6608 - loss: 0.7206 - val_binary_accuracy: 0.6800 - val_loss: 0.6831
Epoch 6/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 613

✓ Test Accuracy: 0.6792 (67.92%)
  Test Loss: 0.6041
  Epochs Trained: 86
  Saved as: ufc_model_exp2_dropout.h5

Experiment 3: Wider Layers (128→64→32)
--------------------------------------------------------------------------------
Epoch 1/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.6025 - loss: 1.9138 - val_binary_accuracy: 0.6704 - val_loss: 1.2259
Epoch 2/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 656us/step - binary_accuracy: 0.6847 - loss: 1.1044 - val_binary_accuracy: 0.6652 - val_loss: 0.8469
Epoch 3/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 646us/step - binary_accuracy: 0.6873 - loss: 0.7980 - val_binary_accuracy: 0.6652 - val_loss: 0.7060
Epoch 4/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 629us/step - binary_accuracy: 0.6929 - loss: 0.6836 - val_binary_accuracy: 0.6704 - val_loss: 0.6531
Epoch 5/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 637us/step - binary_accuracy: 0.6924 - loss: 0.6402 - val_binary_accuracy: 0.6713 - val_loss: 0.6320
Epoch 6/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 640us/step -

✓ Test Accuracy: 0.6808 (68.08%)
  Test Loss: 0.6008
  Epochs Trained: 200
  Saved as: ufc_model_exp3_wide.h5

COMPARISON SUMMARY

Rank   Model Name                          Test Accuracy   Test Loss    Filename                      
----------------------------------------------------------------------------------------------------
🏆 1    Experiment 1: Simple (3 layers)      68.20%        0.6005     ufc_model_exp1_simple.h5      
   2    Experiment 3: Wide Layers            68.08%        0.6008     ufc_model_exp3_wide.h5        
   3    Experiment 2: With Dropout           67.92%        0.6041     ufc_model_exp2_dropout.h5     
   4    Original Model (9 layers)            67.84%        0.6397     ufc_prediction_model.h5       

🏆 BEST MODEL: Experiment 1: Simple (3 layers)
   Accuracy: 68.20%
   Saved as: ufc_model_exp1_simple.h5

💡 TIP: All models are saved with unique names. Your original model is safe!
   To use a different model, just load it:
   model = load_model('ufc_model_exp1

In [28]:
"""
PREDICTION CELL - Copy this into a new Jupyter notebook cell
Automatically finds and uses your best model to predict fight outcomes
"""

# ============================================================================
# AUTO-DETECT BEST MODEL AND SETUP PREDICTION FUNCTION
# ============================================================================

from tensorflow.keras.models import load_model
import os
import glob

# Method 1: Check if best_model variable exists from experimentation cell
try:
    if 'best_model' in globals() and best_model is not None:
        print("✓ Using best_model from experimentation cell")
        model_to_use = best_model
        model_name = best_model_name if 'best_model_name' in globals() else "best_model (from experiments)"
    else:
        raise NameError("best_model not found")
except:
    # Method 2: Find all experiment models and test them
    print("Searching for available models...")
    
    # List of possible model files (in order of preference)
    # Experiment 1 (Simple) is the best, so prioritize it
    model_files = [
        'ufc_model_exp1_simple.h5',  # Experiment 1 - Best model!
        'ufc_model_exp3_wide.h5',
        'ufc_model_exp2_dropout.h5',
        'ufc_prediction_model.h5',  # Original model
        'best_ufc_model.h5',  # From experiment_models.py script
    ]
    
    # Also check for any experiment models
    experiment_models = glob.glob('ufc_model_exp*.h5')
    model_files.extend(experiment_models)
    
    # Remove duplicates while preserving order
    seen = set()
    model_files = [f for f in model_files if f not in seen and not seen.add(f)]
    
    # Find existing models
    existing_models = [f for f in model_files if os.path.exists(f)]
    
    if not existing_models:
        print("❌ Error: No model files found!")
        print("   Make sure you've trained at least one model.")
        raise FileNotFoundError("No model files found")
    
    print(f"Found {len(existing_models)} model(s):")
    for m in existing_models:
        print(f"  - {m}")
    
    # If we have X_test_scaled and y_test, evaluate all and pick best
    try:
        if 'X_test_scaled' in globals() and 'y_test' in globals():
            print("\nEvaluating models to find the best one...")
            best_accuracy = 0
            best_model_file = None
            
            for model_file in existing_models:
                try:
                    test_model = load_model(model_file)
                    results = test_model.evaluate(X_test_scaled, y_test, verbose=0)
                    accuracy = results[1]
                    print(f"  {model_file}: {accuracy*100:.2f}% accuracy")
                    
                    if accuracy > best_accuracy:
                        best_accuracy = accuracy
                        best_model_file = model_file
                        model_to_use = test_model
                except Exception as e:
                    print(f"  ⚠ Could not evaluate {model_file}: {e}")
                    continue
            
            if best_model_file:
                print(f"\n✓ Best model: {best_model_file} ({best_accuracy*100:.2f}% accuracy)")
                model_name = best_model_file
            else:
                # Fallback to first available model
                model_to_use = load_model(existing_models[0])
                model_name = existing_models[0]
                print(f"\n✓ Using: {model_name}")
        else:
            # No test data available, prioritize Experiment 1 (best model)
            if 'ufc_model_exp1_simple.h5' in existing_models:
                model_to_use = load_model('ufc_model_exp1_simple.h5')
                model_name = 'ufc_model_exp1_simple.h5'
                print(f"\n✓ Using Experiment 1 (Simple) - Best model!")
            elif 'ufc_model_exp3_wide.h5' in existing_models:
                model_to_use = load_model('ufc_model_exp3_wide.h5')
                model_name = 'ufc_model_exp3_wide.h5'
            elif 'ufc_model_exp2_dropout.h5' in existing_models:
                model_to_use = load_model('ufc_model_exp2_dropout.h5')
                model_name = 'ufc_model_exp2_dropout.h5'
            else:
                model_to_use = load_model(existing_models[0])
                model_name = existing_models[0]
            print(f"\n✓ Using: {model_name}")
    except Exception as e:
        print(f"⚠ Could not auto-detect best model: {e}")
        print(f"   Using first available: {existing_models[0]}")
        model_to_use = load_model(existing_models[0])
        model_name = existing_models[0]

# Store as best_model for consistency
best_model = model_to_use
print(f"✓ Model loaded and ready: {model_name}\n")

# Load scaler (should already exist from training)
import pickle
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
print("✓ Loaded scaler")

# Make sure df_fighters is loaded (should be from earlier cells)
# If not, uncomment:
# df_fighters = pd.read_csv('fighters_clean.csv', index_col='Name')

print("✓ Ready for predictions!\n")

# ============================================================================
# PREDICTION FUNCTION
# ============================================================================

def predict_fight(fighter1, fighter2):
    """
    Predict fight outcome between two fighters.
    
    Args:
        fighter1: Name of first fighter (Red corner)
        fighter2: Name of second fighter (Blue corner)
    
    Returns:
        Probability of fighter1 winning (0-1)
    """
    # Get numeric stats (same columns used in training)
    stats = [col for col in df_fighters.columns if df_fighters[col].dtype in ['int64', 'float64']]
    
    # Handle Height: conversion if needed (same as training)
    if 'Height:' in df_fighters.columns:
        if 'Height:_inches' not in df_fighters.columns:
            def height_to_inches(height_str):
                if pd.isna(height_str) or height_str == '':
                    return np.nan
                try:
                    parts = str(height_str).replace('"', '').split("'")
                    if len(parts) == 2:
                        feet = float(parts[0].strip())
                        inches = float(parts[1].strip())
                        return feet * 12 + inches
                    return np.nan
                except:
                    return np.nan
            df_fighters['Height:_inches'] = df_fighters['Height:'].apply(height_to_inches)
        
        if 'Height:_inches' in df_fighters.columns and 'Height:_inches' not in stats:
            stats.append('Height:_inches')
        if 'Height:' in stats:
            stats.remove('Height:')
    
    # Check if fighters exist
    if fighter1 not in df_fighters.index:
        print(f"❌ Error: '{fighter1}' not found in database")
        return None
    if fighter2 not in df_fighters.index:
        print(f"❌ Error: '{fighter2}' not found in database")
        return None
    
    # Get fighter stats
    data1 = df_fighters.loc[fighter1, stats]
    data2 = df_fighters.loc[fighter2, stats]
    
    # Calculate difference (fighter1 - fighter2)
    data_diff = (data1 - data2).values.reshape(1, -1)
    
    # Scale using the same scaler from training
    data_diff_scaled = scaler.transform(data_diff)
    
    # Predict
    prediction = best_model.predict(data_diff_scaled, verbose=0)
    probability = prediction[0][0]
    
    return probability

# ============================================================================
# EXAMPLE PREDICTIONS
# ============================================================================

print("="*70)
print("EXAMPLE PREDICTIONS")
print("="*70)

# Example 1
fighter1 = "Kyoji Horiguchi"
fighter2 = "Amir Albazi"
prob = predict_fight(fighter1, fighter2)
if prob is not None:
    print(f"\n{fighter1} vs {fighter2}")
    print(f"  {fighter1} win probability: {prob:.2%}")
    print(f"  {fighter2} win probability: {(1-prob):.2%}")
    winner = fighter1 if prob > 0.5 else fighter2
    confidence = prob if prob > 0.5 else (1-prob)
    print(f"  🏆 Predicted Winner: {winner} ({confidence:.2%} confidence)")

# Example 2
fighter1 = "Jon Jones"
fighter2 = "Daniel Cormier"
prob = predict_fight(fighter1, fighter2)
if prob is not None:
    print(f"\n{fighter1} vs {fighter2}")
    print(f"  {fighter1} win probability: {prob:.2%}")
    print(f"  {fighter2} win probability: {(1-prob):.2%}")
    winner = fighter1 if prob > 0.5 else fighter2
    confidence = prob if prob > 0.5 else (1-prob)
    print(f"  🏆 Predicted Winner: {winner} ({confidence:.2%} confidence)")

print("\n" + "="*70)
print("💡 TIP: Use predict_fight('Fighter 1', 'Fighter 2') to predict any fight!")
print(f"💡 Currently using model: {model_name}")
print("="*70)


✓ Using best_model from experimentation cell
✓ Model loaded and ready: ufc_model_exp1_simple.h5

✓ Loaded scaler
✓ Ready for predictions!

EXAMPLE PREDICTIONS

Kyoji Horiguchi vs Amir Albazi
  Kyoji Horiguchi win probability: 69.84%
  Amir Albazi win probability: 30.16%
  🏆 Predicted Winner: Kyoji Horiguchi (69.84% confidence)

Jon Jones vs Daniel Cormier
  Jon Jones win probability: 89.27%
  Daniel Cormier win probability: 10.73%
  🏆 Predicted Winner: Jon Jones (89.27% confidence)

💡 TIP: Use predict_fight('Fighter 1', 'Fighter 2') to predict any fight!
💡 Currently using model: ufc_model_exp1_simple.h5


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [26]:
fighter1 = "Dustin Jacoby"
fighter2 = "Julius Walker"
prob = predict_fight(fighter1, fighter2)

if prob:
    winner = fighter1 if prob > 0.5 else fighter2
    confidence = prob if prob > 0.5 else (1-prob)
    print(f"\n{fighter1} vs {fighter2}")
    print(f"🏆 Predicted Winner: {winner} ({confidence:.1%} confidence)")


Dustin Jacoby vs Julius Walker
🏆 Predicted Winner: Julius Walker (50.6% confidence)


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
